# Lab: Relationships and Correlations

This lab covers the techniques from Lesson 5 — scatter plots, regression overlays,
pair plots, and correlation heatmaps — using the retail sales dataset.

Work through each section in order. Write your answers in the empty markdown cells.

**Outputs are cleared.** Run each cell to generate the plots.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install matplotlib seaborn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.0)

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "date": pd.date_range("2023-01-01", periods=n, freq="D"),
    "category": np.random.choice(["Electronics", "Clothing", "Books", "Home"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "sales": np.random.lognormal(mean=6.5, sigma=0.8, size=n).round(2),
    "units": np.random.randint(1, 20, n),
    "customer_age": np.random.normal(38, 10, n).clip(18, 70).astype(int),
    "profit_margin": np.random.beta(3, 7, n).round(3),
})

print(df.shape)
df.head()

## Step 1: Scatter plot with regression line

Use `sns.regplot` to explore the relationship between `units` sold and `sales` value.
You might expect more units to mean higher sales — let's see if the data agrees.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=df, x="units", y="sales", ax=ax,
    scatter_kws={"alpha": 0.25, "s": 15},
    line_kws={"color": "crimson"}
)
ax.set_title("Sales vs. units sold")
ax.set_xlabel("Units sold")
ax.set_ylabel("Sales (£)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** Is there a visible positive relationship between units and sales?
> How wide is the confidence band — does it suggest a strong or weak relationship?
> Does this match your expectation?

*(Write your answer here.)*

## Step 2: Linear vs. LOWESS smoother

Compare a linear regression fit against a LOWESS (non-parametric) fit for
`customer_age` vs. `profit_margin`. Plot them side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

common_kws = dict(
    data=df, x="customer_age", y="profit_margin",
    scatter_kws={"alpha": 0.2, "s": 12},
    line_kws={"color": "crimson"}
)

sns.regplot(**common_kws, ax=axes[0])
axes[0].set_title("Linear fit")

sns.regplot(**common_kws, lowess=True, ax=axes[1])
axes[1].set_title("LOWESS (non-parametric)")

for ax in axes:
    ax.set_xlabel("Customer age")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Profit margin")
fig.suptitle("Profit margin vs. customer age — linear vs. LOWESS", fontsize=13)
fig.tight_layout()
plt.show()

> **Question:** Does the LOWESS fit reveal a non-linear pattern that the linear
> fit misses? If the two fits look similar, what does that tell you about the
> underlying relationship?

*(Write your answer here.)*

## Step 3: Faceted scatter with lmplot

Use `sns.lmplot` to show the `units` vs. `sales` relationship separately
for each `region`. Do all regions show the same trend?

In [ ]:
g = sns.lmplot(
    data=df, x="units", y="sales", hue="region",
    scatter_kws={"alpha": 0.2, "s": 12},
    height=5, aspect=1.3
)
g.set_axis_labels("Units sold", "Sales (£)")
g.figure.suptitle("Units vs. sales — by region", y=1.02)
plt.show()

> **Question:** Are the regression lines for each region roughly parallel,
> or do some regions show a steeper or flatter trend?
> Is the confidence band similar across regions, or wider for some?

*(Write your answer here.)*

## Step 4: Pair plot

Build a pair plot of all four numeric variables, coloured by `category`.
Use `corner=True` to show only the lower triangle.

In [ ]:
numeric_cols = ["sales", "units", "customer_age", "profit_margin"]

g = sns.pairplot(
    df[numeric_cols + ["category"]],
    hue="category",
    corner=True,
    plot_kws={"alpha": 0.3, "s": 12},
    diag_kind="kde"
)
g.figure.suptitle("Pairwise relationships — all numeric variables by category", y=1.01)
plt.show()

> **Question:** Which pair of variables shows the strongest visual relationship?
> Are there any pairs where the category groupings are clearly separated?
> Do any of the diagonal KDE plots suggest a bimodal distribution?

*(Write your answer here.)*

## Step 5: Correlation heatmap

Compute the Pearson correlation matrix for all four numeric variables and
display it as a heatmap. Show only the lower triangle.
Use a diverging colourmap centred at zero.

In [ ]:
numeric_cols = ["sales", "units", "customer_age", "profit_margin"]
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, mask=mask, square=True,
    linewidths=0.5, ax=ax
)
ax.set_title("Pearson correlation matrix")
fig.tight_layout()
plt.show()

> **Question:** Which pair has the highest absolute correlation?
> Are any correlations surprising? Pick one cell and look at its scatter plot
> from the pair plot above — does the correlation value match what you see visually?

*(Write your answer here.)*

## Challenge: correlation vs. causation

The checklist from Lesson 5 says: "Correlation near zero doesn't mean no relationship
— always verify with a scatter plot."

Create a new variable that has a clear non-linear relationship with `sales`
(hint: try squaring or taking the log of an existing column), then:

1. Compute its Pearson correlation with `sales`.
2. Plot a scatter of the two variables.
3. Comment on what the correlation coefficient alone would have told you — and what it missed.

In [ ]:
# Step 1: create a transformed variable
df["log_sales"] = np.log(df["sales"])

# Step 2: compute correlation
print("Correlation (sales vs log_sales):",
      df[["sales", "log_sales"]].corr().loc["sales", "log_sales"].round(3))

# Step 3: scatter plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["log_sales"], df["sales"], alpha=0.25, s=15, color="#4c72b0")
ax.set_xlabel("log(Sales)")
ax.set_ylabel("Sales (£)")
ax.set_title("Sales vs. log(Sales) — clearly non-linear")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** What is the Pearson correlation between `sales` and `log_sales`?
> Does the scatter plot reveal something the correlation number doesn't capture?
> What lesson does this reinforce about relying on correlation alone?

*(Write your answer here.)*